In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from shapely.ops import unary_union
import geopandas as gpd

C:\Users\x12la\AppData\Local\Temp\ipykernel_25456\2974714796.py:6: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


In [2]:
cmaq_file = "all_202301_base_reduced.nc"

grid_d02_file = "latlon_ChicagoLADCO_d03.nc"
grid_d03_file = "latlon_ChicagoLADCO_d03.nc"

epa_dir = Path("./")

start_dt = pd.Timestamp("2023-01-01 00:00")
end_dt   = pd.Timestamp("2023-01-31 23:00")

epa_code = ['42401','42602','44201','42101','88101']
var      = ['SO2','NO2','O3','CO','PM25_TOT']

# CMAP county shapefile
cmap_cty = gpd.read_file('C:/Users/x12la/Desktop/Scripts/CMAP_cty.shp')
cmap_cty = cmap_cty.to_crs('EPSG:4326')

In [3]:
ds = xr.open_dataset(cmaq_file)
print(ds)

grid_d02 = xr.open_dataset(grid_d02_file)
grid_d03 = xr.open_dataset(grid_d03_file)

lon_d02 = grid_d02["lon"].values
lat_d02 = grid_d02["lat"].values

lon_d03 = grid_d03["lon"].values
lat_d03 = grid_d03["lat"].values

llat, ulat = lat_d02.min(), lat_d02.max()
llon, ulon = lon_d02.min(), lon_d02.max()

<xarray.Dataset>
Dimensions:   (TSTEP: 744, LAY: 1, ROW: 288, COL: 315)
Dimensions without coordinates: TSTEP, LAY, ROW, COL
Data variables: (12/15)
    SO2       (TSTEP, LAY, ROW, COL) float32 ...
    NO2       (TSTEP, LAY, ROW, COL) float32 ...
    NO        (TSTEP, LAY, ROW, COL) float32 ...
    O3        (TSTEP, LAY, ROW, COL) float32 ...
    CO        (TSTEP, LAY, ROW, COL) float32 ...
    PM25_TOT  (TSTEP, LAY, ROW, COL) float32 ...
    ...        ...
    RH        (TSTEP, LAY, ROW, COL) float32 ...
    SFC_TMP   (TSTEP, LAY, ROW, COL) float32 ...
    PBLH      (TSTEP, LAY, ROW, COL) float32 ...
    precip    (TSTEP, LAY, ROW, COL) float32 ...
    U10       (TSTEP, LAY, ROW, COL) float32 ...
    V10       (TSTEP, LAY, ROW, COL) float32 ...
Attributes: (12/34)
    IOAPI_VERSION:  ioapi-3.2: $Id: init3.F90 247 2023-03-22 15:59:19Z coats ...
    EXEC_ID:        ????????????????                                         ...
    FTYPE:          1
    CDATE:          2026135
    CTIME:  

In [4]:
cmap_union = cmap_cty.unary_union

def label_in_d03(df, lon_col="Longitude", lat_col="Latitude"):
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326"
    )
    return gdf.geometry.within(cmap_union).to_numpy()

In [5]:
def find_index(stn_lon, stn_lat, wrf_lon, wrf_lat):
    xx, yy = [], []
    for i in range(len(stn_lat)):
        abslat = np.abs(wrf_lat - stn_lat[i])
        abslon = np.abs(wrf_lon - stn_lon[i])
        c = np.maximum(abslon, abslat)
        args = np.where(c == np.min(c))
        xx.append(args[-2][0])
        yy.append(args[-1][0])
    return xx, yy

def convert_epa_units(df, species):
    """
    Convert EPA AQS observation units to match CMAQ units.
    """

    if "Units of Measure" not in df.columns:
        return df
    # Only convert gases where EPA might report ppm
    if species in ["O3", "CO", "NO2", "SO2"]:
        mask = df["Units of Measure"].str.contains("Parts per million", na=False)
        # Convert ppm → ppb
        df.loc[mask, "Sample Measurement"] *= 1000
    return df

def crop_epa(file):
    df = pd.read_csv(file)

    # Handle underscore headers in some AQS downloads per Stacy's original code
    if "Date_GMT" in df.columns:
        df.columns = [c.replace("_", " ") for c in df.columns]

    # coarse bbox crop (d02 bbox)
    df = df[(df["Latitude"]  >= llat) & (df["Latitude"]  <= ulat) &
            (df["Longitude"] >= llon) & (df["Longitude"] <= ulon)]

    # Build datetime and time-crop
    df["Datetime GMT"] = pd.to_datetime(df["Date GMT"] + " " + df["Time GMT"])
    df = df[(df["Datetime GMT"] >= start_dt) & (df["Datetime GMT"] <= end_dt)]
    return df

# adopted from Stacy's original code
def merge_species(species, epa_code):
    epa_file = epa_dir / f"hourly_{epa_code}_2023.csv"
    df = crop_epa(epa_file)
    df = convert_epa_units(df, species)

    # Unique station locations
    stations = df[["Latitude", "Longitude"]].drop_duplicates()

    dataframes = []
    t_index = pd.date_range(start_dt, end_dt, freq="1H")

    for _, row in stations.iterrows():
        stn_lat = row["Latitude"]
        stn_lon = row["Longitude"]

        tmp = df[(df.Latitude == stn_lat) & (df.Longitude == stn_lon)].copy()
        tmp.index = tmp["Datetime GMT"]

        # Resample numeric vs metadata exactly like your original script logic
        num_cols = tmp.select_dtypes(include="number").columns
        non_num_cols = tmp.select_dtypes(exclude="number").columns

        tmp_num = tmp[num_cols].resample("1H").mean().reindex(t_index)
        tmp_non = tmp[non_num_cols].resample("1H").first()

        tmp = pd.concat([tmp_num, tmp_non], axis=1)

        # Find nearest d02 grid cell for CMAQ extraction
        x, y = find_index([stn_lon], [stn_lat], lon_d02, lat_d02)
        x, y = x[0], y[0]

        # Basic bounds check (avoids index errors)
        if not (0 <= x < lon_d02.shape[0] and 0 <= y < lon_d02.shape[1]):
            continue

        # Pull CMAQ surface layer
        model = ds[species][:, 0, x, y].values
        tmp["CMAQ"] = model[:len(tmp)]

        # Observations
        tmp["OBS"] = tmp["Sample Measurement"]

        # Metadata for later grouping/filtering
        tmp["Species"] = species
        tmp["x"] = x
        tmp["y"] = y
        tmp["Latitude"] = stn_lat
        tmp["Longitude"] = stn_lon

        dataframes.append(tmp)
    return pd.concat(dataframes, ignore_index=False)

def calc_metrics(df):
    df = df.dropna(subset=["OBS", "CMAQ"])
    if len(df) == 0:
        return pd.Series({
            "N": 0,
            "Mean_OBS": np.nan,
            "Mean_CMAQ": np.nan,
            "MB": np.nan,
            "RMSE": np.nan,
            "R": np.nan,
            "NMB_%": np.nan,
            "NME_%": np.nan,
        })

    obs = df["OBS"].astype(float)
    mod = df["CMAQ"].astype(float)

    diff = mod - obs
    sum_obs = obs.sum()

    # Normalized metrics (percent), guard against divide-by-zero
    if sum_obs == 0:
        nmb = np.nan
        nme = np.nan
    else:
        nmb = 100.0 * diff.sum() / sum_obs
        nme = 100.0 * np.abs(diff).sum() / sum_obs

    return pd.Series({
        "N": len(df),
        "Mean_OBS": obs.mean(),
        "Mean_CMAQ": mod.mean(),
        "MB": diff.mean(),
        "RMSE": np.sqrt((diff ** 2).mean()),
        "R": obs.corr(mod),
        "NMB_%": nmb,
        "NME_%": nme,
    })


In [6]:

combined = []

for i in range(len(var)):
    species = var[i]
    code = epa_code[i]
    print("Processing", species)
    df_i = merge_species(species, code)

    # Ensure Datetime GMT is a column (not index) and remove duplicate columns
    if "Datetime GMT" not in df_i.columns:
        df_i = df_i.reset_index().rename(columns={"index": "Datetime GMT"})
    else:
        # just in case Datetime GMT is also in the index from earlier steps
        df_i = df_i.reset_index(drop=True)

    df_i = df_i.loc[:, ~df_i.columns.duplicated()].copy()
    combined.append(df_i)

combined_df = pd.concat(combined, ignore_index=True)

combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()].copy()
combined_df["Datetime GMT"] = pd.to_datetime(combined_df["Datetime GMT"], errors="coerce")

combined_df["in_d03"] = label_in_d03(combined_df)

print("Datetime GMT columns:", (combined_df.columns == "Datetime GMT").sum())
print("Rows in d03:", combined_df["in_d03"].sum())

Processing SO2


C:\Users\x12la\AppData\Local\Temp\ipykernel_25456\831649441.py:27: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Processing NO2


C:\Users\x12la\AppData\Local\Temp\ipykernel_25456\831649441.py:27: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Processing O3


...

Processing CO


C:\Users\x12la\AppData\Local\Temp\ipykernel_25456\831649441.py:27: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Processing PM25_TOT


C:\Users\x12la\AppData\Local\Temp\ipykernel_25456\831649441.py:27: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Datetime GMT columns: 1
Rows in d03: 14136


In [7]:
hourly_stats_d03 = (
    combined_df[combined_df["in_d03"]]
    .groupby("Species")
    .apply(calc_metrics)
)

In [8]:
d03_hourly = combined_df[combined_df["in_d03"]].copy()

# IMPORTANT: remove duplicate column names before resample/select
d03_hourly = d03_hourly.loc[:, ~d03_hourly.columns.duplicated()].copy()

# Parse datetime safely
d03_hourly["Datetime GMT"] = pd.to_datetime(d03_hourly["Datetime GMT"], errors="coerce")
d03_hourly = d03_hourly.dropna(subset=["Datetime GMT"]).sort_values("Datetime GMT")

# Sanity check
assert "OBS" in d03_hourly.columns and "CMAQ" in d03_hourly.columns, d03_hourly.columns
assert (d03_hourly.columns == "OBS").sum() == 1
assert (d03_hourly.columns == "CMAQ").sum() == 1

daily_df_d03 = (
    d03_hourly
    .set_index("Datetime GMT")
    .groupby(["Species", "Latitude", "Longitude"])[["OBS", "CMAQ"]]
    .resample("1D")
    .mean()
    .reset_index()
)

daily_stats_d03 = (
    daily_df_d03
    .groupby("Species")
    .apply(calc_metrics)
)


In [9]:
month_tag = start_dt.strftime("%Y%m")

combined_df.to_csv(f"CMAQ_EPA_hourly_allstations_{month_tag}.csv", index=False)

# d03-only outputs
d03_hourly.to_csv(f"CMAQ_EPA_hourly_d03_{month_tag}.csv", index=False)
daily_df_d03.to_csv(f"CMAQ_EPA_daily_d03_{month_tag}.csv", index=False)

hourly_stats_d03.to_csv(f"Hourly_metrics_d03_{month_tag}.csv")
daily_stats_d03.to_csv(f"Daily_metrics_d03_{month_tag}.csv")

print("Done.")
print(hourly_stats_d03)
print(daily_stats_d03)

Done.
               N    Mean_OBS   Mean_CMAQ         MB        RMSE         R  \
Species                                                                     
CO        1465.0  263.541980  185.480512 -78.061467  122.632129  0.461856   
NO2       3621.0   15.743966   14.512213  -1.231753    8.849837  0.489003   
O3         725.0   18.245517   26.911896   8.666379   10.887482  0.761970   
PM25_TOT  5894.0   10.416949   11.209708   0.792758    5.541374  0.656963   
SO2       2119.0    0.188910    0.492319   0.303410    1.396893  0.128531   

               NMB_%       NME_%  
Species                           
CO        -29.620126   37.023421  
NO2        -7.823653   42.915003  
O3         47.498673   52.357938  
PM25_TOT    7.610272   37.582160  
SO2       160.610768  552.635891  
              N    Mean_OBS   Mean_CMAQ         MB       RMSE         R  \
Species                                                                   
CO         62.0  263.666885  185.275096 -78.391790  93.5163

In [10]:
# ============================================================
# DEBUG WHERE CMAP O3 MONITORS ARE LOST
# ============================================================

o3_file = "hourly_44201_2023.csv"
raw_o3 = pd.read_csv(o3_file)

cmap_counties = ["Cook", "DuPage", "Kane", "Kendall", "Lake", "McHenry", "Will"]

raw_o3_cmap = raw_o3[
    (raw_o3["State Name"] == "Illinois") &
    (raw_o3["County Name"].isin(cmap_counties))
].copy()

print("Raw Illinois CMAP O3 stations:",
      raw_o3_cmap[["County Name", "Site Num", "Latitude", "Longitude"]]
      .drop_duplicates().shape[0])

display(
    raw_o3_cmap[["County Name", "Site Num", "Latitude", "Longitude"]]
    .drop_duplicates()
    .sort_values(["County Name", "Site Num"])
)

# Check date filtering
raw_o3_cmap["datetime"] = pd.to_datetime(
    raw_o3_cmap["Date GMT"] + " " + raw_o3_cmap["Time GMT"],
    errors="coerce"
)

jan_o3_cmap = raw_o3_cmap[
    (raw_o3_cmap["datetime"] >= start_dt) &
    (raw_o3_cmap["datetime"] <= end_dt)
].copy()

print("\nAfter January date filter:")
print("Rows:", len(jan_o3_cmap))
print("Stations:",
      jan_o3_cmap[["County Name", "Site Num", "Latitude", "Longitude"]]
      .drop_duplicates().shape[0])

display(
    jan_o3_cmap[["County Name", "Site Num", "Latitude", "Longitude"]]
    .drop_duplicates()
    .sort_values(["County Name", "Site Num"])
)

# Compare with final combined_df
final_o3 = combined_df[
    (combined_df["Species"] == "O3") &
    (combined_df["State Name"] == "Illinois") &
    (combined_df["County Name"].isin(cmap_counties))
].copy()

print("\nFinal O3 in combined_df, Illinois CMAP counties:")
print("Rows:", len(final_o3))
print("Stations:",
      final_o3[["County Name", "Site Num", "Latitude", "Longitude"]]
      .drop_duplicates().shape[0])

display(
    final_o3[["County Name", "Site Num", "Latitude", "Longitude", "in_d03"]]
    .drop_duplicates()
    .sort_values(["County Name", "Site Num"])
)

# See which stations disappeared
jan_sites = jan_o3_cmap[["County Name", "Site Num", "Latitude", "Longitude"]].drop_duplicates()
final_sites = final_o3[["County Name", "Site Num", "Latitude", "Longitude"]].drop_duplicates()

missing_after_processing = jan_sites.merge(
    final_sites,
    on=["County Name", "Site Num", "Latitude", "Longitude"],
    how="left",
    indicator=True
)

missing_after_processing = missing_after_processing[
    missing_after_processing["_merge"] == "left_only"
].drop(columns="_merge")

print("\nStations present after January filter but missing from combined_df:")
print("Count:", len(missing_after_processing))

display(
    missing_after_processing.sort_values(["County Name", "Site Num"])
)

C:\Users\x12la\AppData\Local\Temp\ipykernel_25456\3536247384.py:6: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_o3 = pd.read_csv(o3_file)


Raw Illinois CMAP O3 stations: 15


,County Name,Site Num,Latitude,Longitude
3028001,Cook,1,41.670992,-87.732457
3033818,Cook,32,41.755832,-87.545350
3039217,Cook,76,41.751400,-87.713488
3045057,Cook,1003,41.984332,-87.792002
3050852,Cook,1601,41.668120,-87.990570
3056470,Cook,3103,41.965193,-87.876265
3062194,Cook,4002,41.855243,-87.752470
3067072,Cook,4007,42.060285,-87.863225
3072948,Cook,4201,42.139996,-87.799227
3081238,Cook,7002,42.062053,-87.675254



After January date filter:
Rows: 725
Stations: 1


,County Name,Site Num,Latitude,Longitude
3072948,Cook,4201,42.139996,-87.799227



Final O3 in combined_df, Illinois CMAP counties:
Rows: 725
Stations: 1


,County Name,Site Num,Latitude,Longitude,in_d03
17862,Cook,4201.0,42.139996,-87.799227,True



Stations present after January filter but missing from combined_df:
Count: 0


,County Name,Site Num,Latitude,Longitude


In [11]:
raw_o3_cmap["datetime"] = pd.to_datetime(
    raw_o3_cmap["Date GMT"] + " " + raw_o3_cmap["Time GMT"],
    errors="coerce"
)

raw_o3_cmap["month"] = raw_o3_cmap["datetime"].dt.month

month_site_counts = (
    raw_o3_cmap
    .groupby(["County Name", "Site Num", "Latitude", "Longitude", "month"])
    .size()
    .reset_index(name="n_obs")
    .sort_values(["County Name", "Site Num", "month"])
)

display(month_site_counts)

site_month_summary = (
    raw_o3_cmap
    .groupby(["County Name", "Site Num", "Latitude", "Longitude"])["month"]
    .agg(lambda x: sorted(x.dropna().unique()))
    .reset_index(name="months_available")
)

display(site_month_summary)

,County Name,Site Num,Latitude,Longitude,month,n_obs
0,Cook,1,41.670992,-87.732457,3,729
1,Cook,1,41.670992,-87.732457,4,715
2,Cook,1,41.670992,-87.732457,5,738
3,Cook,1,41.670992,-87.732457,6,710
4,Cook,1,41.670992,-87.732457,7,733
...,...,...,...,...,...,...
133,Will,1011,41.221537,-88.190967,7,735
134,Will,1011,41.221537,-88.190967,8,685
135,Will,1011,41.221537,-88.190967,9,720
136,Will,1011,41.221537,-88.190967,10,744


,County Name,Site Num,Latitude,Longitude,months_available
0,Cook,1,41.670992,-87.732457,"[3, 4, 5, 6, 7, 8, 9, 10, 11]"
1,Cook,32,41.755832,-87.545350,"[3, 4, 5, 6, 7, 8, 9, 10, 11]"
2,Cook,76,41.751400,-87.713488,"[3, 4, 5, 6, 7, 8, 9, 10, 11]"
3,Cook,1003,41.984332,-87.792002,"[3, 4, 5, 6, 7, 8, 9, 10, 11]"
4,Cook,1601,41.668120,-87.990570,"[3, 4, 5, 6, 7, 8, 9, 10, 11]"
5,Cook,3103,41.965193,-87.876265,"[3, 4, 5, 6, 7, 8, 9, 10, 11]"
6,Cook,4002,41.855243,-87.752470,"[3, 4, 5, 6, 7, 8, 9, 10, 11]"
7,Cook,4007,42.060285,-87.863225,"[3, 4, 5, 6, 7, 8, 9, 10, 11]"
8,Cook,4201,42.139996,-87.799227,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
9,Cook,7002,42.062053,-87.675254,"[3, 4, 5, 6, 7, 8, 9, 10, 11]"
